In [1]:
import os
from pathlib import Path

ROOT_DIR_BRAINTREEBANK = Path(r"C:\Users\simon\PyCharmMiscProject\neuroprobe-dev\braintreebank")
os.environ["ROOT_DIR_BRAINTREEBANK"] = str(ROOT_DIR_BRAINTREEBANK)

assert ROOT_DIR_BRAINTREEBANK.exists(), ROOT_DIR_BRAINTREEBANK

In [2]:
CHECKPOINT_DIR = Path("notebooks/models")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = CHECKPOINT_DIR / "leaderboard_week3_best.pt"
print("Checkpoint will be saved to:", CHECKPOINT_PATH)

Checkpoint will be saved to: notebooks\models\leaderboard_week3_best.pt


In [3]:
# Standard library
import copy
import inspect
import json
import random

# Third-party
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


# Scikit-learn
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Neuroprobe
import neuroprobe
import neuroprobe.train_test_splits as neuroprobe_train_test_splits
from neuroprobe import (
    BrainTreebankSubject,
    BrainTreebankSubjectTrialBenchmarkDataset,
    generate_splits_cross_session,
    generate_splits_cross_subject,
    generate_splits_within_session,
)

In [4]:
# =========================
# Config
# =========================
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TASKS = ["delta_volume", "speech", "pitch", "gpt2_surprisal", "word_gap"]

BATCH_SIZE = 8
BASE_LR = 2e-4
WEIGHT_DECAY = 1e-3
NUM_EPOCHS = 20
PATIENCE = 5
NUM_WORKERS = 0

STFT_N_FFT = 64
STFT_HOP = 16
STFT_WIN_LEN = 32
LAPLACIAN_K = 4

SUBJ_EMB_DIM = 16
TASK_EMB_DIM = 8

COORD_DIM = 3
COORD_EMB_DIM = 16
ELEC_HIDDEN_DIM = 128
MODEL_DIM = 128

NUM_VIRTUAL_SENSORS = 16
NUM_SENSOR_HEADS = 4
ATTN_DROPOUT = 0.1
PROJ_DROPOUT = 0.1

USE_COORDS_IN_KEYS = True
USE_COORDS_IN_VALUES = False
USE_SENSOR_SELF_ATTN = True
NUM_SENSOR_SELF_ATTN_LAYERS = 1

TEST_SUBJECT_ID = 1
TEST_TRIAL_ID = 2
SUBJECT_IDS = list(range(1, 11))

USE_GLOBAL_TRAIN_NORM = False
USE_VAL_AS_TEST = False

MASK_VALUE = -1e9

print("DEVICE:", DEVICE)

DEVICE: cpu


In [5]:
# =========================
# Subjects
# =========================
def build_all_subjects(subject_ids):
    all_subjects = {}
    for sid in subject_ids:
        all_subjects[int(sid)] = BrainTreebankSubject(
            subject_id=int(sid),
            cache=True,
            dtype=torch.float32,
            coordinates_type="cortical",
        )
    return all_subjects


ALL_SUBJECTS = build_all_subjects(SUBJECT_IDS)

print("ALL_SUBJECTS keys:", sorted(ALL_SUBJECTS.keys()))
print({k: type(v).__name__ for k, v in ALL_SUBJECTS.items()})

ALL_SUBJECTS keys: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
{1: 'BrainTreebankSubject', 2: 'BrainTreebankSubject', 3: 'BrainTreebankSubject', 4: 'BrainTreebankSubject', 5: 'BrainTreebankSubject', 6: 'BrainTreebankSubject', 7: 'BrainTreebankSubject', 8: 'BrainTreebankSubject', 9: 'BrainTreebankSubject', 10: 'BrainTreebankSubject'}


In [6]:
# =========================
# Electrode geometry scan
# =========================
rows = []

for subject_id in SUBJECT_IDS:
    subject = ALL_SUBJECTS[subject_id]

    print(f"\nsubject {subject_id}: type={type(subject).__name__}")
    print("subject attrs:", [a for a in dir(subject) if "trial" in a.lower()][:20])

    trial_dict = getattr(subject, "subject_trials", None)
    if trial_dict is None:
        print(f"subject {subject_id}: no subject_trials attribute")
        continue

    trial_ids = sorted(trial_dict.keys())
    print(f"subject {subject_id}: trial_ids={trial_ids}")

    for trial_id in trial_ids:
        try:
            trial = trial_dict[trial_id]
            print(f"  trial {trial_id}: type={type(trial).__name__}")

            if hasattr(trial, "electrode_coordinates"):
                coords = np.asarray(trial.electrode_coordinates)
                coord_source = "electrode_coordinates"
            elif hasattr(trial, "electrode_locs"):
                coords = np.asarray(trial.electrode_locs)
                coord_source = "electrode_locs"
            else:
                print(
                    f"  trial {trial_id}: no coord attr; attrs={[a for a in dir(trial) if 'electrode' in a.lower()][:20]}")
                continue

            print(f"  trial {trial_id}: {coord_source} shape={getattr(coords, 'shape', None)}")

            if coords.ndim != 2 or coords.shape[1] < 3:
                print(f"  trial {trial_id}: bad coord shape {coords.shape}")
                continue

            coords = coords[:, :3]
            n_elec = coords.shape[0]

            center = coords.mean(axis=0)
            centered = coords - center
            radii = np.linalg.norm(centered, axis=1)

            if n_elec > 1:
                d = np.sqrt(((coords[:, None, :] - coords[None, :, :]) ** 2).sum(axis=-1))
                d[d == 0] = np.inf
                nn_dist = d.min(axis=1)
                nn_mean = float(nn_dist.mean())
                nn_median = float(np.median(nn_dist))
            else:
                nn_mean = np.nan
                nn_median = np.nan

            rows.append({
                "subject_id": subject_id,
                "trial_id": trial_id,
                "n_elec": n_elec,
                "x_std": float(coords[:, 0].std()),
                "y_std": float(coords[:, 1].std()),
                "z_std": float(coords[:, 2].std()),
                "radius_mean": float(radii.mean()),
                "radius_std": float(radii.std()),
                "nn_mean": nn_mean,
                "nn_median": nn_median,
            })

        except Exception as e:
            print(f"skip subject={subject_id} trial={trial_id}: {e}")

geom_df = pd.DataFrame(rows)

print("\nrows collected:", len(rows))
print("geom_df columns:", list(geom_df.columns))

if geom_df.empty:
    print("geom_df is empty")
else:
    display(geom_df.sort_values(["subject_id", "trial_id"]))
    print(geom_df["n_elec"].describe())

KeyError: 'subject_id'

In [ ]:
rows = []

for subject_id in SUBJECT_IDS:
    subject = ALL_SUBJECTS[subject_id]
    trial_ids = sorted(subject.subject_trials.keys()) if hasattr(subject, "subject_trials") else []

    for trial_id in trial_ids:
        try:
            trial = subject.subject_trials[trial_id]

            if hasattr(trial, "electrode_coordinates"):
                coords = np.asarray(trial.electrode_coordinates)
            elif hasattr(trial, "electrode_locs"):
                coords = np.asarray(trial.electrode_locs)
            else:
                continue

            if coords.ndim != 2 or coords.shape[1] < 3:
                continue

            coords = coords[:, :3]
            n_elec = coords.shape[0]

            center = coords.mean(axis=0)
            centered = coords - center
            radii = np.linalg.norm(centered, axis=1)

            if n_elec > 1:
                d = np.sqrt(((coords[:, None, :] - coords[None, :, :]) ** 2).sum(axis=-1))
                d[d == 0] = np.inf
                nn_dist = d.min(axis=1)
                nn_mean = float(nn_dist.mean())
                nn_median = float(np.median(nn_dist))
            else:
                nn_mean = np.nan
                nn_median = np.nan

            rows.append({
                "subject_id": subject_id,
                "trial_id": trial_id,
                "n_elec": n_elec,
                "x_std": float(coords[:, 0].std()),
                "y_std": float(coords[:, 1].std()),
                "z_std": float(coords[:, 2].std()),
                "radius_mean": float(radii.mean()),
                "radius_std": float(radii.std()),
                "nn_mean": nn_mean,
                "nn_median": nn_median,
            })
        except Exception as e:
            print(f"skip subject={subject_id} trial={trial_id}: {e}")

geom_df = pd.DataFrame(rows)
display(geom_df.sort_values(["subject_id", "trial_id"]))
print(geom_df["n_elec"].describe())